In [1]:
import numpy as np
from scipy.sparse import load_npz

X_train = load_npz('../data/features/X_priority_train.npz')
X_val = load_npz('../data/features/X_priority_val.npz')
X_test = load_npz('../data/features/X_priority_test.npz')

y_train = np.load('../data/features/y_priority_train.npy')
y_val = np.load('../data/features/y_priority_val.npy')
y_test = np.load('../data/features/y_priority_test.npy')

print(X_train.shape, y_train.shape)

(140080, 122) (140080,)


In [2]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor

In [3]:
from sklearn.metrics import classification_report, accuracy_score

y_val_pred = xgb_model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

Validation Accuracy: 0.24431818181818182

Classification Report:
              precision    recall  f1-score   support

           0       0.24      0.24      0.24      7565
           1       0.24      0.24      0.24      7471
           2       0.25      0.24      0.24      7394
           3       0.25      0.26      0.25      7490

    accuracy                           0.24     29920
   macro avg       0.24      0.24      0.24     29920
weighted avg       0.24      0.24      0.24     29920



In [4]:
import joblib
joblib.dump(xgb_model, '../saved_models/priority_classifier_baseline.pkl')

['../saved_models/priority_classifier_baseline.pkl']

In [5]:
import torch

X_train_dense = torch.FloatTensor(X_train.toarray())
X_val_dense = torch.FloatTensor(X_val.toarray())
X_test_dense = torch.FloatTensor(X_test.toarray())

y_train_tensor = torch.LongTensor(y_train)
y_val_tensor = torch.LongTensor(y_val)
y_test_tensor = torch.LongTensor(y_test)

print(X_train_dense.shape)

torch.Size([140080, 122])


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/features/featured_data.csv')

# Separate text, structured and targets
text_data = df['issue_description']
y = df['priority']

structured_data = df.drop(columns=[
    'issue_description', 'priority', 
    'sla_breached', 'customer_satisfaction_score'
])

print(text_data.shape)
print(structured_data.shape)

(200000,)
(200000, 20)


In [7]:
from torch.nn.utils.rnn import pad_sequence
import torch

# Build vocabulary
vocab = {}
idx = 1  # 0 reserved for padding

for text in text_data:
    for word in str(text).split():
        if word not in vocab:
            vocab[word] = idx
            idx += 1

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 79


In [8]:
# Convert text to sequences
def text_to_sequence(text, vocab):
    return [vocab.get(word, 0) for word in str(text).split()]

sequences = [text_to_sequence(text, vocab) for text in text_data]

# Check sequence lengths
lengths = [len(s) for s in sequences]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)/len(lengths):.1f}")

Min: 9, Max: 13, Mean: 11.0


In [9]:
# Pad sequences
max_len = 13

padded_sequences = np.array([
    seq + [0] * (max_len - len(seq)) for seq in sequences
])

print(padded_sequences.shape)

(200000, 13)


In [10]:
print(structured_data.dtypes)

product                          str
category                         str
channel                          str
region                           str
customer_age                   int64
customer_gender                  str
subscription_type                str
customer_tenure_months         int64
previous_tickets               int64
first_response_time_hours    float64
escalated                        str
operating_system                 str
browser                          str
payment_method                   str
language                         str
preferred_contact_time           str
customer_segment                 str
day_of_week                    int64
month                          int64
ticket_rate                  float64
dtype: object


In [11]:
from sklearn.model_selection import train_test_split

# Convert text sequences and targets to arrays
padded_sequences = np.array(padded_sequences)

# Label encode target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split all three together
X_text_tv, X_text_test, X_struct_tv, X_struct_test, y_tv, y_test = train_test_split(
    padded_sequences, structured_data, y_encoded, test_size=0.15, random_state=42
)

X_text_train, X_text_val, X_struct_train, X_struct_val, y_train, y_val = train_test_split(
    X_text_tv, X_struct_tv, y_tv, test_size=0.176, random_state=42
)

print(X_text_train.shape, X_struct_train.shape, y_train.shape)

(140080, 13) (140080, 20) (140080,)


In [12]:
import joblib

# Load saved artifacts
ohe_priority = joblib.load('../saved_models/ohe_priority.pkl')
scaler_priority = joblib.load('../saved_models/scaler_priority.pkl')

# Drop leaky columns for priority goal
X_struct_train = X_struct_train.drop(columns=['first_response_time_hours', 'escalated'])
X_struct_val = X_struct_val.drop(columns=['first_response_time_hours', 'escalated'])
X_struct_test = X_struct_test.drop(columns=['first_response_time_hours', 'escalated'])

# Categorical columns
cat_cols = [col for col in X_struct_train.select_dtypes('object').columns]

# One-hot encode
def encode_cats(X, ohe, cat_cols):
    encoded = ohe.transform(X[cat_cols])
    encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cat_cols), index=X.index)
    return pd.concat([X.drop(columns=cat_cols), encoded_df], axis=1)

X_struct_train = encode_cats(X_struct_train, ohe_priority, cat_cols)
X_struct_val = encode_cats(X_struct_val, ohe_priority, cat_cols)
X_struct_test = encode_cats(X_struct_test, ohe_priority, cat_cols)

print(X_struct_train.shape)


/tmp/ipykernel_12581/648955826.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = [col for col in X_struct_train.select_dtypes('object').columns]


(140080, 72)


In [13]:
num_cols = ['customer_age', 'customer_tenure_months', 'previous_tickets',
            'day_of_week', 'month', 'ticket_rate']

X_struct_train[num_cols] = scaler_priority.transform(X_struct_train[num_cols])
X_struct_val[num_cols] = scaler_priority.transform(X_struct_val[num_cols])
X_struct_test[num_cols] = scaler_priority.transform(X_struct_test[num_cols])

# Convert to tensors
import torch

X_text_train_tensor = torch.LongTensor(X_text_train)
X_text_val_tensor = torch.LongTensor(X_text_val)
X_text_test_tensor = torch.LongTensor(X_text_test)

X_struct_train_tensor = torch.FloatTensor(X_struct_train.values)
X_struct_val_tensor = torch.FloatTensor(X_struct_val.values)
X_struct_test_tensor = torch.FloatTensor(X_struct_test.values)

y_train_tensor = torch.LongTensor(y_train)
y_val_tensor = torch.LongTensor(y_val)
y_test_tensor = torch.LongTensor(y_test)

print(X_text_train_tensor.shape, X_struct_train_tensor.shape)

torch.Size([140080, 13]) torch.Size([140080, 72])


In [14]:
import torch.nn as nn

class DualInputModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, struct_input_dim, num_classes):
        super(DualInputModel, self).__init__()
        
        # Text branch - LSTM
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        
        # Structured branch - MLP
        self.struct_fc = nn.Sequential(
            nn.Linear(struct_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Combined classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + 64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, text, struct):
        # Text branch
        embedded = self.embedding(text)
        _, (hidden, _) = self.lstm(embedded)
        text_out = hidden[-1]
        
        # Structured branch
        struct_out = self.struct_fc(struct)
        
        # Combine
        combined = torch.cat([text_out, struct_out], dim=1)
        return self.classifier(combined)

# Initialize model
model = DualInputModel(
    vocab_size=len(vocab),
    embed_dim=32,
    hidden_dim=64,
    struct_input_dim=72,
    num_classes=4
)

print(model)

DualInputModel(
  (embedding): Embedding(80, 32, padding_idx=0)
  (lstm): LSTM(32, 64, batch_first=True)
  (struct_fc): Sequential(
    (0): Linear(in_features=72, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=4, bias=True)
  )
)


In [15]:
from torch.utils.data import TensorDataset, DataLoader

# Dataset and dataloader
train_dataset = TensorDataset(X_text_train_tensor, X_struct_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_text_val_tensor, X_struct_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 2189
Val batches: 468


In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = model.to(device)

num_epochs = 5

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for text_batch, struct_batch, y_batch in train_loader:
        text_batch = text_batch.to(device)
        struct_batch = struct_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(text_batch, struct_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(y_batch).sum().item()
        total += y_batch.size(0)
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for text_batch, struct_batch, y_batch in val_loader:
            text_batch = text_batch.to(device)
            struct_batch = struct_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(text_batch, struct_batch)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(y_batch).sum().item()
            val_total += y_batch.size(0)
    
    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Train Acc: {correct/total:.4f} | "
          f"Val Acc: {val_correct/val_total:.4f}")

Using device: cpu
Epoch 1/5 | Train Loss: 1.3868 | Train Acc: 0.2500 | Val Acc: 0.2502
Epoch 2/5 | Train Loss: 1.3865 | Train Acc: 0.2502 | Val Acc: 0.2517
Epoch 3/5 | Train Loss: 1.3864 | Train Acc: 0.2522 | Val Acc: 0.2487
Epoch 4/5 | Train Loss: 1.3863 | Train Acc: 0.2522 | Val Acc: 0.2514
Epoch 5/5 | Train Loss: 1.3862 | Train Acc: 0.2533 | Val Acc: 0.2504


In [17]:
# Save model
torch.save(model.state_dict(), '../saved_models/priority_classifier_dl.pth')

# Test evaluation
model.eval()
test_correct = 0
test_total = 0

test_dataset = TensorDataset(X_text_test_tensor, X_struct_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

with torch.no_grad():
    for text_batch, struct_batch, y_batch in test_loader:
        text_batch = text_batch.to(device)
        struct_batch = struct_batch.to(device)
        y_batch = y_batch.to(device)
        outputs = model(text_batch, struct_batch)
        _, predicted = outputs.max(1)
        test_correct += predicted.eq(y_batch).sum().item()
        test_total += y_batch.size(0)

print(f"Test Accuracy: {test_correct/test_total:.4f}")

Test Accuracy: 0.2523


In [18]:
joblib.dump(vocab, '../saved_models/priority_vocab.pkl')

['../saved_models/priority_vocab.pkl']